# Intro to Machine Learning: Training & Inference

We'll teach a neural network to learn the shape of a **sine wave** from noisy data — completely from scratch.

### The big idea
1. **Data** — show the model many (x, y) examples
2. **Model** — a neural network that takes x and predicts y
3. **Training** — repeatedly adjust the model's parameters to reduce its mistakes
4. **Inference** — use the trained model to predict on new, unseen inputs

In [ ]:
%pip install torch numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries ready.")

## Step 1 — The Data

We generate 200 points along a sine wave and add random **noise** to simulate real-world measurements.

We split them into two groups **before** training starts:
- **Training set (80%)** — the model learns from these
- **Test set (20%)** — held back and only used to check how well the model generalised

In [ ]:
N = 200
X_np = np.linspace(-2 * np.pi, 2 * np.pi, N).astype(np.float32)
noise = np.random.normal(0, 0.3, N).astype(np.float32)
y_np = np.sin(X_np) + noise

# 80/20 train/test split
split = int(N * 0.8)
indices = np.random.permutation(N)
train_idx, test_idx = indices[:split], indices[split:]

X_train_np, y_train_np = X_np[train_idx], y_np[train_idx]
X_test_np,  y_test_np  = X_np[test_idx],  y_np[test_idx]

# PyTorch tensors
X_train = torch.tensor(X_train_np).unsqueeze(1)
y_train = torch.tensor(y_train_np).unsqueeze(1)
X_test  = torch.tensor(X_test_np).unsqueeze(1)
y_test  = torch.tensor(y_test_np).unsqueeze(1)

print(f"Training points: {len(X_train_np)}  |  Test points: {len(X_test_np)}")

# Plot — show which points are train vs test
fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(X_train_np, y_train_np, s=14, alpha=0.7, color="steelblue", label=f"Training data ({len(X_train_np)} points)")
ax.scatter(X_test_np,  y_test_np,  s=14, alpha=0.9, color="orange",    marker="^", label=f"Test data ({len(X_test_np)} points) — hidden from model during training")
ax.plot(X_np, np.sin(X_np), color="red", linewidth=2, linestyle="--", label="True sine wave (hidden from model)")
ax.set_title("Dataset split: training vs test")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

## Step 2 — The Model

A neural network is made of **layers**. Each layer transforms numbers — like a chain of mathematical functions.

```
Input (x)  →  [Layer 1: 64 neurons]  →  [Layer 2: 64 neurons]  →  Output (predicted y)
```

- **Linear** layers multiply inputs by learned weights and add a bias
- **ReLU** is an activation function that adds non-linearity, letting the network learn curves (not just straight lines)
- The more layers/neurons, the more complex patterns the model can learn

In [ ]:
class SineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(1, 64),   # 1 input  → 64 neurons
            nn.ReLU(),
            nn.Linear(64, 64),  # 64       → 64 neurons
            nn.ReLU(),
            nn.Linear(64, 1),   # 64       → 1 output
        )

    def forward(self, x):
        return self.network(x)


model = SineNet()
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal learnable parameters: {total_params}")

## Step 3 — Training

Training is a loop that repeats hundreds of times:

1. **Forward pass** — feed x into the model, get a prediction
2. **Compute loss** — measure how wrong the prediction is (we use Mean Squared Error: average of (prediction - actual)²)
3. **Backward pass** — calculate how to adjust each parameter to reduce the loss (this is called *backpropagation*)
4. **Update** — the optimizer nudges each parameter in the right direction (we use *Adam*, a popular optimizer)

We'll snapshot the model's predictions at epochs 0, 50, 200, and 1000 to watch it learn.

In [ ]:
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 1000
loss_history = []
snapshots = {}  # epoch → predictions

for epoch in range(EPOCHS + 1):
    # Forward pass — training data only
    pred = model(X_train)
    loss = loss_fn(pred, y_train)

    if epoch in (0, 50, 200, EPOCHS):
        with torch.no_grad():
            snapshots[epoch] = model(X_train).squeeze().numpy()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 200 == 0:
        print(f"Epoch {epoch:>4}  |  Loss: {loss.item():.4f}")

print("\nTraining complete.")

## Step 4 — Visualise the Loss Curve

The **loss** is the model's error score. As training progresses it should drop toward zero — meaning the model is getting better at predicting the sine wave.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_history, color="steelblue", linewidth=1.5)
ax.set_title("Training Loss over Time")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_yscale("log")

for epoch in (50, 200, 1000):
    ax.axvline(epoch, color="orange", linestyle="--", alpha=0.6)
    ax.text(epoch + 10, max(loss_history) * 0.4, f"epoch {epoch}", color="orange", fontsize=9)

plt.tight_layout()
plt.show()

## Step 5 — Watch the Model Learn

Here are four snapshots of what the model predicted at different stages of training. You can see it start as a flat random line and gradually sculpt itself into the sine wave shape.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (epoch, preds) in zip(axes, snapshots.items()):
    ax.scatter(X_train_np, y_train_np, s=10, alpha=0.4, color="steelblue", label="Training data")
    ax.plot(np.sort(X_train_np), np.sin(np.sort(X_train_np)), color="red", linewidth=1.5, linestyle="--", label="True sine")
    ax.plot(np.sort(X_train_np), preds[np.argsort(X_train_np)], color="orange", linewidth=2, label="Model prediction")
    ax.set_title(f"Epoch {epoch}  |  Loss: {loss_history[epoch]:.4f}")
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8)

fig.suptitle("Model Learning Progress", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 6 — Inference

Now the model is trained. We run it on the **test set** — the 20% of points it never saw during training.

This is the honest measure of how well it learned: can it predict correctly on data it hasn't memorised?

In [ ]:
model.eval()

with torch.no_grad():
    y_pred = model(X_test).squeeze().numpy()

test_loss = loss_fn(torch.tensor(y_pred).unsqueeze(1), y_test).item()
print(f"Test loss (MSE): {test_loss:.4f}")

# Sort test points by x so the prediction line draws cleanly
sort_order = np.argsort(X_test_np)
X_sorted   = X_test_np[sort_order]
y_sorted   = y_pred[sort_order]

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_train_np, y_train_np, s=12, alpha=0.3, color="steelblue", label="Training data (seen by model)")
ax.scatter(X_test_np,  y_test_np,  s=20, alpha=0.9, color="orange", marker="^", label="Test data (never seen during training)")
ax.plot(X_np, np.sin(X_np), color="red", linewidth=2, linestyle="--", label="True sine wave")
ax.plot(X_sorted, y_sorted, color="green", linewidth=2.5, label=f"Model predictions on test set  (MSE: {test_loss:.4f})")
ax.set_title("Inference on held-out test data")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Concept | What it means |
|---|---|
| **Data** | The examples the model learns from (x → y pairs) |
| **Model** | A neural network — a chain of layers with learnable parameters |
| **Loss** | A score of how wrong the model is (lower = better) |
| **Training** | Looping: predict → measure loss → backpropagate → update weights |
| **Inference** | Using the trained model to predict on new inputs |
| **Generalisation** | How well the model works *outside* the training data (purple region above) |

> **Notice** the purple shaded region: the model was never trained on those x values, yet it still approximates the sine wave reasonably well. That ability to generalise is what makes neural networks useful in the real world.